# SLM Phase Optimization Workflow

## Overview
This notebook generates and optimizes phase patterns for Spatial Light Modulators (SLM), supporting microlens array design and optimization.

## Workflow
1. **Generate Job List** - Configure optimization parameters via GUI interface and create batch jobs
2. **Run Optimization Jobs** - Execute batch phase optimization and save results to `./output/`
3. **View Results** - Browse and visualize optimization results, upload directly to SLM

## Output Files
Each job generates files in `./output/{job_title}/`:
- `{job_title}.npy` - 8-bit phase pattern (can be directly uploaded to SLM)
- `{job_title}.json` - Optimization parameters record
- `{job_title}_optimizer.pkl` - Complete optimizer object (for subsequent visualization)

---

# Step 1: Generate Job List

In [ ]:
"""
Step 1: Load Configuration and Launch GUI
==========================================
Features:
- Load default optical parameters from JSON config file
- Launch interactive GUI for configuring and managing optimization jobs

GUI Usage:
- Left panel: Set optical parameters (M, focal length, overlap ratio, etc.)
- Right panel: Manage job list
- Click "Add to Job List" to add current configuration to job queue
- Supports batch adding multiple jobs with different parameters
"""

from optics_utils import load_dict_from_json
from phase_optimizer_gui import create_optimizer_gui
import os

# ============================================================
# Config File Selection
# ============================================================
# Base config file (contains default optical parameters)
filename_base = r"base.json"
# Can also load previously saved optimization config:
# filename_base = r"251105_1_super_0.9.json"

# Config file directory
path_json = r".\\config\\"

# ============================================================
# Load Config and Launch GUI
# ============================================================
# Load JSON config file
params = load_dict_from_json(os.path.join(path_json, filename_base))

# Create optimizer GUI
gui = create_optimizer_gui(default_params=params)

# Step 2: Run Optimization Jobs

Execute all jobs added in the GUI. Three running modes supported:
- **Simulation mode** (`sim_mode=True`): Only perform optimization calculations, no hardware connection
- **Remote mode** (`remote_mode=True`): Connect to remote SLM via RPyC
- **Local mode**: Directly connect to local SLM

> **Note**: Optimization requires GPU acceleration, ensure CUDA is available

In [2]:
"""
Step 2: Batch Execute Optimization Jobs
========================================
Features:
- Iterate through job list from GUI and execute each optimization
- Save optimization results to ./output/ directory
- Supports Fresnel mode (direct generation) and Optimized mode (gradient optimization)

Output:
- Each job generates an independent folder containing .npy, .json, .pkl files
"""

from batch_processor import process_jobs
from hardware import RemoteSLMManager, SLMManager
import torch

# ============================================================
# Device Selection: Prefer GPU (CUDA)
# ============================================================
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

# ============================================================
# SLM Connection Mode Selection
# ============================================================
# Mode description:
#   sim_mode=True  : Simulation mode, no hardware connection (recommended for pure optimization)
#   remote_mode=True: Remote mode, connect to remote SLM service via RPyC
#   Both False     : Local mode, directly connect to local SLM hardware

remote_mode = True   # Whether to use remote SLM
sim_mode = True      # Whether to use simulation mode (no hardware)

# Create SLM manager based on mode
if sim_mode:
    # Simulation mode: No actual SLM connection, only optimization calculations
    slm_manager = SLMManager(sim_mode=True)
elif remote_mode:
    # Remote mode: Connect to remote SLM service (requires RPyC server running)
    slm_manager = RemoteSLMManager()
else: 
    # Local mode: Directly connect to local SLM hardware
    slm_manager = SLMManager(sim_mode=False)

# ============================================================
# Execute Batch Optimization
# ============================================================
# process_jobs parameters:
#   gui: GUI instance containing job list
#   slm_manager: SLM manager instance
#   device: Computation device (cuda/cpu)
#   output_dir: Output directory (default './output')
#   save_optimizer: Whether to save optimizer object (default True)
#   upsampling: Upsampling factor during optimization (default 2.0)

results = process_jobs(gui, slm_manager=slm_manager, device=device)

# results is a dictionary with job names as keys, values contain:
#   - 'status': 'success' or 'error'
#   - 'output_dir': Output directory path
#   - 'npy_path': Phase pattern file path
#   - 'json_path': Parameters file path
#   - 'optimizer_path': Optimizer object path
#   - 'optimizer': Optimizer instance (for further analysis)

Using device: cuda
🎬 Simulation Mode Enabled
🚀 Starting batch processing: 21 job(s)
📁 Output directory: ./output
🖥️  Device: cuda


[1/21] Processing: M5_depth_range_1.0x_airy1.0_over0.25_rand0.05
--------------------------------------------------
  Mode: Optimized
  Airy correction: 1.0
  Depth in focus: [-0.5, 0.5]
Using device: cuda
PSF randomization enabled: randomness=0.05, seed=42
F/37.85, 73.9 mm
Max. Lens width: 1.953mm
Lens width (Fresnel): 1.562mm
Max. Overlap: 25.0 %

Airy radius: 23.8um
Airy radius (fresnel): 29.7um
Airy correction for Fresnel: 1.250
Depth of focus: 1.48mm
Depth of focus (fresnel): 2.31mm
DOF correction for Fresnel: 1.56

Multi-depth planes are used at F=73.2 (-0.5DOF), 74.6 (0.5DOF) mm
Starting optimization with 500 iterations...
Iter: 1/500
  Focal_mse: 1946.7513 Depth_mse: 1946.7441 Eff_mean: -0.0108 Eff_std: 0.0083 Eff_mean (unweighted):
  0.0005 Eff_std (unweighted): 0.0002 Total: 3.8935e+03
Iter: 51/500
  Focal_mse: 156.2724 Depth_mse: 182.7273 Eff_me

# Step 3: View and Manage Results

Use interactive browser to view saved optimization results:
- **Select Job**: Click a job in the list to view details
- **Visualize**: Click "Visualize" button to generate visualization charts
- **Upload to SLM**: If SLM is connected, can directly upload phase pattern

Visualization includes:
- Phase pattern distribution
- Target PSF vs. actual PSF comparison
- Energy distribution and focusing efficiency

In [3]:
"""
Step 3: Browse and Visualize Optimization Results
==================================================
Features:
- Scan ./output/ directory for all saved jobs
- Provide interactive GUI for browsing and selecting jobs
- Load optimizer objects and generate visualization charts
- Support direct phase pattern upload to SLM

GUI Legend:
- Check mark indicates job contains complete optimizer object (.pkl)
- Circle indicates job only has phase pattern and parameter files
"""

from batch_processor import browse_jobs

# ============================================================
# Create Results Browser
# ============================================================
# browse_jobs parameters:
#   output_dir: Directory to search for jobs (default './output')
#   upsampling: Upsampling factor for visualization (default 3)
#   slm_manager: Optional, if provided enables direct SLM upload

browser = browse_jobs(output_dir='./output')

# ============================================================
# Get Currently Selected Optimizer (optional)
# ============================================================
# After selecting and visualizing a job in GUI, get optimizer object:
# optimizer = browser.get_current_optimizer()

# Get currently selected phase pattern:
# phase_8bit = browser.get_current_phase()

# Get job list:
# job_list = browser.get_job_list()

✅ Found 45 job(s) in output
📁 Job Browser - Click a job to view details, then Visualize
   ✓ = has optimizer, ○ = no optimizer




Output()